In [ ]:
# ===== 셀 1: 준비물 가져오기 (도구 불러오기) =====

import os                              # 컴퓨터의 환경설정(.env 값 등)을 읽기 위한 도구
import requests                        # 인터넷 주소에 데이터를 요청(다운로드)하는 도구
import pandas as pd                    # 표(엑셀 같은 형태)로 데이터를 다루는 도구. 보통 'pd'라는 별명으로 씀
from dotenv import load_dotenv         # .env 파일을 읽어오는 도구
# (이 API는 JSON으로 받기 때문에 xml 도구는 필요 없습니다)

load_dotenv()                          # .env 파일을 읽어서 안의 값들을 사용할 수 있게 준비
SERVICE_KEY = os.getenv("PUBLIC_CITY_PARK_KEY")   # .env 안의 PUBLIC_CITY_PARK_KEY 값을 꺼내서 변수에 저장

print("서비스 키 로드됨:", bool(SERVICE_KEY))   # 키를 잘 불러왔는지 확인 (True면 성공)
if not SERVICE_KEY:
    print("⚠️ .env에 PUBLIC_CITY_PARK_KEY 가 없습니다. 위 마크다운 안내대로 추가 후 커널을 재시작하세요.")

In [ ]:
# ===== 셀 2: 전국 도시공원 데이터를 인터넷에서 모두 가져오기 =====
# (컬럼명 한글 변환은 하지 않고 원본 컬럼명 그대로 가져옵니다)
#
# 공공데이터포털 표준데이터 주소 형식 (KOPIS처럼 ?serviceKey=...&... 쿼리 방식):
#   http://api.data.go.kr/openapi/tn_pubr_public_cty_park_info_api?serviceKey=...&type=json&pageNo=1&numOfRows=100

BASE_URL = "http://api.data.go.kr/openapi/tn_pubr_public_cty_park_info_api"   # 도시공원 표준데이터 주소
NUM_ROWS = 1000                          # 한 번에 가져올 건수 (한 페이지 최대)

rows = []                                # 가져온 공원들을 담아둘 빈 목록(리스트)
page = 1                                 # 현재 페이지 번호 (1부터 시작)
total = None                             # 전체 건수 (첫 응답에서 알아냄)

while True:                              # 더 이상 데이터가 없을 때까지 반복
    params = {                           # 서버에 보낼 조회 조건들
        "serviceKey": SERVICE_KEY,       # 내 인증키 (신분증 역할)
        "type": "json",                  # 응답 형식: JSON으로 받기
        "pageNo": page,                  # 가져올 페이지 번호
        "numOfRows": NUM_ROWS,           # 한 페이지 건수
    }
    res = requests.get(BASE_URL, params=params)   # 위 조건으로 서버에 데이터 요청

    # 혹시 JSON이 아닌(에러 HTML 등) 응답이 오면 원인을 바로 보여주고 멈춤
    try:
        body = res.json()["response"]["body"]   # 표준데이터 응답에서 본문(body)만 꺼냄
    except Exception:
        print("❌ JSON 파싱 실패 — 응답 앞부분을 확인하세요:")
        print("status:", res.status_code)
        print(res.text[:500])
        raise

    if total is None:                    # 첫 응답이라면
        total = int(body.get("totalCount", 0))   # 전체 건수를 기억해 둠

    items = body.get("items", [])        # 이번 페이지의 실제 데이터 목록
    if isinstance(items, dict):          # 가끔 {"item": [...]} 형태로 한 겹 더 감싸진 경우 대비
        items = items.get("item", [])
    rows += items                        # 전체 목록에 이어 붙이기

    if len(rows) >= total or not items:  # 다 모았거나 더 줄 게 없으면 끝
        break
    page += 1                            # 아니면 다음 페이지로

df = pd.DataFrame(rows)                  # 모은 데이터를 표(df)로 만듦 (컬럼명은 원본 그대로)

print(f"전체 건수(API 기준): {total}건")    # API가 알려준 전체 건수
print(f"실제 가져온 데이터: {len(df)}건 ({page}페이지)")   # 우리가 실제로 모은 건수

In [ ]:
# ===== 셀 3: 가져온 데이터 확인하기 =====

print("[컬럼 목록]")
print(list(df.columns))               # 어떤 컬럼들이 있는지 (원본 컬럼명) 확인

print("\n[표 크기] (행, 열):", df.shape)   # 데이터가 몇 행 몇 열인지 확인

print("\n[데이터 정보]")
df.info()                             # 각 컬럼의 타입과 빈 값 여부 등 요약 정보

In [ ]:
# ===== 셀 4: 상위 10줄 미리보기 =====

df.head(10)   # df 표의 맨 위 10줄을 그대로 보여줌 (모든 컬럼)